# 07. Synthetic pretrain V1

Результаты записываются в `outputs/`, а модели — в `checkpoints/`.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("неверный путь.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from project_paths import *
ensure_project_directories()

print("Project root:", PROJECT_ROOT)

Project root: D:\Users\user\Desktop\DS_XRD_project


# Пре-трейн голов XRD — v1

**Данные**: 460 133 синтетических спектра (COD + crystalDB, генератор v4) на сетке (2, 4096):
канал 1 — sqrt-нормированная интенсивность, канал 2 — пер-точечная маска [0°, 90°].
Conditioning: [λ1, λ2, есть_ли_дублет].

**Головы** (все с масками):
| Голова | Тип | Выход |
|---|---|---|
| lattice (log a, log b, log c, α, β, γ) | regression, SmoothL1 | 6 |
| cell volume (log V) — auxiliary | regression, SmoothL1 | 1 |
| space group | classification | 230 |
| crystal system | classification | 7 |
| elements presence | multi-label, BCE | словарь из синтетики |

**Режимы** (переменная MODE в следующей ячейке):
- `sanity` — оверфит 64 спектров, 250 шагов (проверка, что модель вообще способна учиться);
- `test` — 120k подвыборка, 1 эпоха (~470 шагов) + валидация (этот режим запускался для проверки пайплайна);
- `full` — все 460k, 12 эпох (~2–3 ч на RTX 3060 Ti).

Железо, под которое подобраны параметры: RTX 3060 Ti 8 ГБ / 32 ГБ ОЗУ / Ryzen 5600X.
Batch 256, AMP fp16, warmup 300 шагов + cosine. DataLoader без воркеров (Windows + memmap = быстро и надёжно).

In [ ]:
import json
import math
import random
import time
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if DEVICE.type == 'cuda':
    print('device:', DEVICE, '|', torch.cuda.get_device_name(0))
    
else:
    print('device:', DEVICE)

device: cuda | NVIDIA GeForce RTX 3060 Ti


In [3]:
MODE = 'full'   # 'sanity' | 'test' | 'full'

BASE = PROJECT_ROOT
DATA = BASE / 'data' / 'preprocessed'
SYNTH_PARQUET = BASE / 'data' / 'clean' / 'df_synth_summary_final_clean.parquet'
FT_POOL = BASE / 'data' / 'clean' / 'ft_pool_combined.parquet'
OUT = BASE / 'outputs'
CKPT_DIR = BASE / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)

GRID_N = 4096
SYSTEMS = ['triclinic', 'monoclinic', 'orthorhombic', 'tetragonal',
           'trigonal', 'hexagonal', 'cubic']

HP = dict(
    batch=256,
    lr=3e-4,
    wd=1e-4,
    warmup=300,
    clip=1.0,
    epochs={'sanity': 1, 'test': 1, 'full': 12}[MODE],
    train_subset={'sanity': 64, 'test': 120_000, 'full': None}[MODE],
    val_subset={'sanity': 64, 'test': 4000, 'full': None}[MODE],
    val_every={'sanity': None, 'test': None, 'full': 2000}[MODE],
    sanity_steps=250,
)
print('MODE =', MODE, '| HP:', HP)

MODE = full | HP: {'batch': 256, 'lr': 0.0003, 'wd': 0.0001, 'warmup': 300, 'clip': 1.0, 'epochs': 12, 'train_subset': None, 'val_subset': None, 'val_every': 2000, 'sanity_steps': 250}


In [ ]:
# ---------- данные: индекс этапа B + метки из синтетического паркета ----------

index = pd.read_parquet(DATA / 'index_preprocessed.parquet')
pre = index[index['split_role'] == 'pretrain'].reset_index(drop=True)
print('pretrain строк:', len(pre))

syn = pd.read_parquet(SYNTH_PARQUET)[
    ['sample_id', 'lattice_a', 'lattice_b', 'lattice_c',
     'alpha', 'beta', 'gamma', 'spacegroup_number',
     'crystal_system', 'elements_list', 'phase_compositions']
]
df = pre.merge(syn, on='sample_id', how='left')
assert df['lattice_a'].notna().all(), 'метки решётки должны быть у всей синтетики'

def to_list(v):
    if isinstance(v, str):
        try:
            return json.loads(v)
        
        except Exception:
            return []
        
    if isinstance(v, (list, tuple, np.ndarray)):
        return list(v)
    
    return []

df['elements'] = df['elements_list'].apply(to_list)
df['formula'] = df['phase_compositions'].apply(
    lambda p: (p[0] if isinstance(p, (list, tuple, np.ndarray)) and len(p) > 0
               else (p if isinstance(p, str) else 'NA'))
)

# словарь элементов

cnt = Counter()

for els in df['elements']:
    cnt.update(els)

VOCAB = sorted(cnt)
EL_IDX = {e: i for i, e in enumerate(VOCAB)}
(OUT / 'element_vocab.json').write_text(json.dumps(VOCAB))

print('элементов в словаре:', len(VOCAB))

# объём ячейки из параметров решётки

abc = df[['lattice_a', 'lattice_b', 'lattice_c']].to_numpy(np.float64)
ang = df[['alpha', 'beta', 'gamma']].to_numpy(np.float64)
ca, cb, cg = (np.cos(np.radians(ang[:, i])) for i in range(3))
t = 1.0 - ca**2 - cb**2 - cg**2 + 2.0 * ca * cb * cg
V = abc[:, 0] * abc[:, 1] * abc[:, 2] * np.sqrt(np.clip(t, 1e-12, None))

df['lat6'] = list(np.hstack([np.log(abc), ang]))
df['logV'] = np.log(V)

print('V: медиана %.0f A^3, p05 %.0f, p95 %.0f' % (np.median(V), np.percentile(V, 5), np.percentile(V, 95)))

pretrain строк: 460133
элементов в словаре: 98
V: медиана 1220 A^3, p05 126, p95 4049


In [ ]:
# ---------- сплит по соединениям (98/2) + стандартизация ----------

rng = np.random.default_rng(SEED)
uniq = df['formula'].unique()
val_formulas = set(rng.choice(uniq, size=max(1, int(len(uniq) * 0.02)), replace=False))
is_val = df['formula'].isin(val_formulas).to_numpy()

train_df = df[~is_val].reset_index(drop=True)
val_df = df[is_val].reset_index(drop=True)

print(f'train {len(train_df)} / val {len(val_df)} | уникальных формул: {len(uniq)}, в вале {len(val_formulas)}')

# статистики стандартизации ТОЛЬКО по train

lat_arr = np.stack(train_df['lat6'].to_numpy())
LAT_MEAN = lat_arr.mean(0)
LAT_STD = lat_arr.std(0) + 1e-6
VOL_MEAN = float(train_df['logV'].mean())
VOL_STD = float(train_df['logV'].std() + 1e-6)

print('log(a) mean/std: %.3f/%.3f | angles mean/std(альфа): %.1f/%.1f'
      % (LAT_MEAN[0], LAT_STD[0], LAT_MEAN[3], LAT_STD[3]))

stats = dict(lat_mean=LAT_MEAN.tolist(), lat_std=LAT_STD.tolist(),
             vol_mean=VOL_MEAN, vol_std=VOL_STD, systems=SYSTEMS, vocab=VOCAB)
(OUT / 'pretrain_stats.json').write_text(json.dumps(stats))

pd.DataFrame({'sample_id': df['sample_id'],
              'split': np.where(is_val, 'val', 'train')}).to_parquet(OUT / 'splits_pretrain.parquet')

print('сплиты и статистики сохранены')

train 451027 / val 9106 | уникальных формул: 300862, в вале 6017
log(a) mean/std: 2.186/0.451 | angles mean/std(альфа): 88.7/12.1
сплиты и статистики сохранены


In [ ]:
# ---------- датасет поверх меммапов ----------

N_TOTAL = len(index)
X_MM = np.memmap(DATA / 'X_intensity.f16', dtype=np.float16, mode='r', shape=(N_TOTAL, GRID_N))
M_MM = np.memmap(DATA / 'M_mask.u8', dtype=np.uint8, mode='r', shape=(N_TOTAL, GRID_N))

class PretrainDS(Dataset):
    def __init__(self, frame):
        self.row = frame['row_idx'].to_numpy(np.int64)
        lam1 = frame['lambda_1'].to_numpy(np.float32)
        lam2 = frame['lambda_2'].to_numpy(np.float32)
        has2 = np.isfinite(lam2).astype(np.float32)
        self.lam = np.stack([lam1 / 1.54, np.nan_to_num(lam2) / 1.54, has2], 1)

        self.lat6 = (np.stack(frame['lat6'].to_numpy()) - LAT_MEAN) / LAT_STD
        self.latm = np.ones(len(frame), np.float32)
        self.vol = ((frame['logV'].to_numpy(np.float32) - VOL_MEAN) / VOL_STD)
        self.volm = np.ones(len(frame), np.float32)

        sg = frame['spacegroup_number'].to_numpy(float)
        self.sgm = np.isfinite(sg).astype(np.float32)
        self.sg = np.nan_to_num(sg).astype(np.int64) - 1  # SG 1..230 -> классы 0..229

        sysmap = {s: i for i, s in enumerate(SYSTEMS)}
        sys_raw = frame['crystal_system'].map(sysmap)
        self.sysm = sys_raw.notna().to_numpy(np.float32)
        self.sys = sys_raw.fillna(0).to_numpy(np.int64)

        n = len(frame)
        self.el = np.zeros((n, len(VOCAB)), np.float32)

        for i, els in enumerate(frame['elements']):
            for e in els:
                j = EL_IDX.get(e)

                if j is not None:
                    self.el[i, j] = 1.0
                    
        self.elm = np.ones(n, np.float32)

    def __len__(self):
        return len(self.row)

    def __getitem__(self, i):
        r = self.row[i]
        x = np.empty((2, GRID_N), np.float32)
        x[0] = X_MM[r]
        x[1] = M_MM[r]

        return (torch.from_numpy(x),
                torch.from_numpy(self.lam[i]),
                torch.from_numpy(self.lat6[i]), torch.tensor(self.latm[i]),
                torch.tensor(self.sg[i]), torch.tensor(self.sgm[i]),
                torch.tensor(self.sys[i]), torch.tensor(self.sysm[i]),
                torch.from_numpy(self.el[i]), torch.tensor(self.elm[i]),
                torch.tensor(self.vol[i]), torch.tensor(self.volm[i]))

def make_loaders():
    tr, va = train_df, val_df

    if HP['train_subset'] and HP['train_subset'] < len(tr):
        tr = tr.sample(HP['train_subset'], random_state=SEED)

    if HP['val_subset'] and HP['val_subset'] < len(va):
        va = va.sample(HP['val_subset'], random_state=SEED)
    ld_tr = DataLoader(PretrainDS(tr), batch_size=HP['batch'], shuffle=True,
                       drop_last=True, pin_memory=True)
    ld_va = DataLoader(PretrainDS(va), batch_size=HP['batch'], shuffle=False, pin_memory=True)

    return ld_tr, ld_va

train_loader, val_loader = make_loaders()
print('train batches:', len(train_loader), '| val batches:', len(val_loader))

train batches: 1761 | val batches: 36


In [ ]:
# ---------- модель: 1D-CNN бэкбон + лямбда-conditioning + 5 голов ----------
class ResBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(cin, cout, 3, stride=stride, padding=1, bias=False)
        self.n1 = nn.GroupNorm(8, cout)
        self.conv2 = nn.Conv1d(cout, cout, 3, padding=1, bias=False)
        self.n2 = nn.GroupNorm(8, cout)

        if cin == cout and stride == 1:
            self.skip = nn.Identity()

        else:
            self.skip = nn.Sequential(nn.Conv1d(cin, cout, 1, stride=stride, bias=False),
                                      nn.GroupNorm(8, cout))

    def forward(self, x):
        h = F.gelu(self.n1(self.conv1(x)))
        h = self.n2(self.conv2(h))
        
        return F.gelu(h + self.skip(x))

class XRDNet(nn.Module):
    def __init__(self, n_el):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(2, 32, 15, padding=7, bias=False),
            nn.GroupNorm(8, 32), nn.GELU())
        chans = [(32, 48), (48, 64), (64, 96),
                 (96, 128), (128, 192), (192, 256)]
        self.blocks = nn.Sequential(*[ResBlock(ci, co, stride=2) for ci, co in chans])
        self.lam_mlp = nn.Sequential(nn.Linear(3, 16), nn.GELU(), nn.Linear(16, 16))
        self.trunk = nn.Sequential(nn.Linear(256 + 16, 512), nn.GELU(),
                                   nn.Linear(512, 512), nn.GELU())
        self.head_lat = nn.Linear(512, 6)
        self.head_vol = nn.Linear(512, 1)
        self.head_sg = nn.Linear(512, 230)
        self.head_sys = nn.Linear(512, 7)
        self.head_el = nn.Linear(512, n_el)

    def forward(self, x, lam):
        f = self.stem(x)          # (B, 32, 4096)
        f = self.blocks(f)        # (B, 256, 64)

        # маскированный average pooling: веса = прореженная маска
        
        w = F.adaptive_avg_pool1d(x[:, 1:2], f.shape[-1]).clamp_min(1e-3)
        pooled = (f * w).sum(-1) / w.sum(-1)
        z = torch.cat([pooled, self.lam_mlp(lam)], dim=1)
        z = self.trunk(z)
        return dict(lat=self.head_lat(z),
                    vol=self.head_vol(z).squeeze(-1),
                    sg=self.head_sg(z),
                    sys=self.head_sys(z),
                    el=self.head_el(z))

model = XRDNet(len(VOCAB)).to(DEVICE)
n_par = sum(p.numel() for p in model.parameters())
print('параметров: %.2f M' % (n_par / 1e6))

параметров: 1.37 M


In [ ]:
# ---------- маскированные лоссы и метрики ----------

def masked_l1(pred, tgt, mask):
    m = mask > 0
    if m.sum() == 0:
        return pred.new_zeros(())
    return F.smooth_l1_loss(pred[m], tgt[m])

def masked_ce(logits, tgt, mask):
    m = mask > 0
    if m.sum() == 0:
        return logits.new_zeros(())
    return F.cross_entropy(logits[m], tgt[m].long())

def masked_bce(logits, tgt):
    return F.binary_cross_entropy_with_logits(logits, tgt)

def compute_losses(out, batch):
    (_, _, lat, latm, sg, sgm, sys_, sysm, el, elm, vol, volm) = batch
    losses = dict(
        lat=masked_l1(out['lat'], lat, latm),
        vol=masked_l1(out['vol'], vol, volm),
        sg=masked_ce(out['sg'], sg, sgm),
        sys=masked_ce(out['sys'], sys_, sysm),
        el=masked_bce(out['el'], el),
    )
    return losses

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    sums = defaultdict(float)
    n = 0
    sg_ok = sg_n = sys_ok = sys_n = 0
    tp = fp = fn = 0
    lat_abs = np.zeros(6)
    lat_n = 0
    vol_rel = []
    for batch in loader:
        batch = [b.to(DEVICE, non_blocking=True) for b in batch]
        x, lam = batch[0], batch[1]
        with torch.autocast('cuda', dtype=torch.float16):
            out = model(x, lam)
        losses = compute_losses(out, batch)
        bs = len(x)
        n += bs
        for k, v in losses.items():
            sums[k] += float(v) * bs
        sums['total'] += float(sum(losses.values())) * bs

        sg_m = batch[5] > 0
        if sg_m.any():
            sg_ok += (out['sg'][sg_m].argmax(1) == batch[4][sg_m].long()).sum().item()
            sg_n += int(sg_m.sum())
        sys_m = batch[7] > 0
        if sys_m.any():
            sys_ok += (out['sys'][sys_m].argmax(1) == batch[6][sys_m].long()).sum().item()
            sys_n += int(sys_m.sum())

        pred_bin = (out['el'] > 0).float()
        tp += float((pred_bin * batch[8]).sum().item())
        fp += float((pred_bin * (1 - batch[8])).sum().item())
        fn += float(((1 - pred_bin) * batch[8]).sum().item())

        latm = batch[3] > 0
        if latm.any():
            p = out['lat'][latm].float().cpu().numpy() * LAT_STD + LAT_MEAN
            t = batch[2][latm].cpu().numpy() * LAT_STD + LAT_MEAN
            p[:, :3] = np.exp(p[:, :3])
            t[:, :3] = np.exp(t[:, :3])
            lat_abs += np.abs(p - t).sum(0)
            lat_n += len(p)
            pv = out['vol'][latm].float().cpu().numpy() * VOL_STD + VOL_MEAN
            tv = batch[10][latm].cpu().numpy() * VOL_STD + VOL_MEAN
            vol_rel.append(np.abs(np.exp(pv) - np.exp(tv)) / np.exp(tv))
    model.train()

    prec = tp / max(tp + fp, 1e-9)
    rec = tp / max(tp + fn, 1e-9)
    res = {k: v / n for k, v in sums.items()}
    res['sg_acc'] = sg_ok / max(sg_n, 1)
    res['sys_acc'] = sys_ok / max(sys_n, 1)
    res['el_f1_micro'] = 2 * prec * rec / max(prec + rec, 1e-9)
    if lat_n:
        mae = lat_abs / lat_n
        res['mae_a_A'] = mae[0]
        res['mae_b_A'] = mae[1]
        res['mae_c_A'] = mae[2]
        res['mae_ang_deg'] = mae[3:].mean()
        res['vol_rel_err'] = float(np.concatenate(vol_rel).mean())
    res['n'] = n
    return res

def fmt_metrics(m):
    keys = ['total', 'lat', 'vol', 'sg', 'sys', 'el',
            'sg_acc', 'sys_acc', 'el_f1_micro',
            'mae_a_A', 'mae_b_A', 'mae_c_A', 'mae_ang_deg', 'vol_rel_err']
    return {k: round(float(m[k]), 4) for k in keys if k in m}

In [ ]:
# ---------- оптимизатор, шедулер, обучение ----------
opt = torch.optim.AdamW(model.parameters(), lr=HP['lr'], weight_decay=HP['wd'])
scaler = torch.amp.GradScaler('cuda')

def make_sched(total_steps):
    def fn(step):
        if step < HP['warmup']:
            return step / max(HP['warmup'], 1)
        p = (step - HP['warmup']) / max(total_steps - HP['warmup'], 1)
        return 0.5 * (1.0 + math.cos(math.pi * min(p, 1.0)))
    return torch.optim.lr_scheduler.LambdaLR(opt, fn)

def move(batch):
    return [b.to(DEVICE, non_blocking=True) for b in batch]

def train_step(batch):
    batch = move(batch)
    with torch.autocast('cuda', dtype=torch.float16):
        out = model(batch[0], batch[1])
        losses = compute_losses(out, batch)
        total = sum(losses.values())
    opt.zero_grad(set_to_none=True)
    scaler.scale(total).backward()
    scaler.unscale_(opt)
    nn.utils.clip_grad_norm_(model.parameters(), HP['clip'])
    scaler.step(opt)
    scaler.update()
    return {k: float(v) for k, v in losses.items()} | {'total': float(total)}

def run_training(tag):
    log_rows = []
    if MODE == 'sanity':

        # оверфит фиксированных 64 спектров: лоссы должны упасть почти до нуля
        
        for g in opt.param_groups:
            g['lr'] = 1e-3
        it = iter(train_loader)
        for step in range(1, HP['sanity_steps'] + 1):
            try:
                batch = next(it)
            except StopIteration:
                it = iter(train_loader)
                batch = next(it)
            ls = train_step(batch)
            if step % 25 == 0 or step == 1:
                print(f'step {step:4d} | total {ls["total"]:.4f} | '
                      f'lat {ls["lat"]:.4f} sg {ls["sg"]:.4f} sys {ls["sys"]:.4f} '
                      f'el {ls["el"]:.4f} vol {ls["vol"]:.4f}', flush=True)
        return log_rows

    total_steps = HP['epochs'] * len(train_loader)
    sched = make_sched(total_steps)

    # подменяем шедулер в train_step через глобальную переменную

    global _sched
    _sched = sched
    best = float('inf')
    t0 = time.time()
    step = 0
    runma = defaultdict(float)

    for epoch in range(HP['epochs']):
        for batch in train_loader:
            ls = train_step(batch)
            _sched.step()
            step += 1

            for k, v in ls.items():
                runma[k] += v

            if step % 50 == 0 or step == 1:
                avg = {k: v / (50 if step > 1 else 1) for k, v in runma.items()}
                runma = defaultdict(float)
                msg = ' | '.join(f'{k} {avg[k]:.4f}' for k in ['total', 'lat', 'sg', 'sys', 'el', 'vol'])
                print(f'epoch {epoch+1} step {step:5d}/{total_steps} | {msg} | '
                      f'{(time.time()-t0)/step:.2f} с/шаг', flush=True)
                log_rows.append(dict(step=step, epoch=epoch + 1, **avg))

            if HP['val_every'] and step % HP['val_every'] == 0:
                m = evaluate(model, val_loader)
                print('  VAL:', fmt_metrics(m), flush=True)
                if m['total'] < best:
                    best = m['total']
                    torch.save(model.state_dict(), CKPT_DIR / f'{tag}_best.pt')

    torch.save(model.state_dict(), CKPT_DIR / f'{tag}_last.pt')
    print(f'готово за {(time.time()-t0)/60:.1f} мин | шагов {step}')
    return log_rows

In [ ]:
# ---------- запуск ----------
torch.manual_seed(SEED)

if MODE == 'sanity':

    # берём только полностью размеченные строки, чтобы все головы работали

    global train_df_sanity, train_loader
    train_loader = DataLoader(PretrainDS(train_df.head(64)), batch_size=16, shuffle=False)
    sanity_log = run_training('sanity')
    print()
    print('SANITY: если total упал на порядки - модель и пайплайн обучаемы')
    
else:
    base_metrics = evaluate(model, val_loader)
    print('БАЗЛАЙН (до обучения, случайная инициализация):')
    print(fmt_metrics(base_metrics))
    print()
    train_log = run_training(f'pretrain_v1_{MODE}')
    final_metrics = evaluate(model, val_loader)
    print()
    print('ИТОГ (после обучения):')
    print(fmt_metrics(final_metrics))

БАЗЛАЙН (до обучения, случайная инициализация):
{'total': 8.9563, 'lat': 0.4, 'vol': 0.4601, 'sg': 5.4426, 'sys': 1.9605, 'el': 0.6931, 'sg_acc': 0.0027, 'sys_acc': 0.0461, 'el_f1_micro': 0.0968, 'mae_a_A': 3.4437, 'mae_b_A': 3.8148, 'mae_c_A': 5.2992, 'mae_ang_deg': 8.077, 'vol_rel_err': 1.8591}

epoch 1 step     1/21132 | total 8.9102 | lat 0.3872 | sg 5.4334 | sys 1.9628 | el 0.6932 | vol 0.4335 | 7.92 с/шаг
epoch 1 step    50/21132 | total 8.3595 | lat 0.3789 | sg 5.1562 | sys 1.7428 | el 0.6596 | vol 0.4220 | 2.10 с/шаг
epoch 1 step   100/21132 | total 6.2914 | lat 0.3818 | sg 3.6048 | sys 1.5109 | el 0.3744 | vol 0.4196 | 2.04 с/шаг
epoch 1 step   150/21132 | total 5.5435 | lat 0.3901 | sg 3.1120 | sys 1.4906 | el 0.1277 | vol 0.4232 | 2.03 с/шаг
epoch 1 step   200/21132 | total 5.5356 | lat 0.3846 | sg 3.1220 | sys 1.4900 | el 0.1141 | vol 0.4250 | 1.98 с/шаг
epoch 1 step   250/21132 | total 5.4450 | lat 0.3799 | sg 3.0934 | sys 1.4633 | el 0.1151 | vol 0.3932 | 1.91 с/шаг
epoch

In [ ]:
# ---------- график лоссов ----------
if MODE != 'sanity':
    lg = pd.DataFrame(train_log)
    fig, ax = plt.subplots(figsize=(9, 4.5))

    for k in ['total', 'lat', 'sg', 'sys', 'el', 'vol']:
        ax.plot(lg['step'], lg[k], label=k, lw=1.6)

    ax.set_xlabel('шаг')
    ax.set_ylabel('loss (running avg)')
    ax.set_yscale('log')
    ax.grid(alpha=0.3)
    ax.legend(ncol=3, fontsize=9)
    ax.set_title(f'Пре-трейн, MODE={MODE}')
    
    plt.tight_layout()
    plt.savefig(OUT / f'pretrain_{MODE}_losses.png', dpi=130)
    print('график:', OUT / f'pretrain_{MODE}_losses.png')
    plt.show()

график: D:\Users\user\Desktop\DS_XRD_project\outputs\pretrain_full_losses.png


D:\UserTemp\ipykernel_19172\3944774684.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# ---------- FT-сплит 95/5 для следующего этапа ----------
from sklearn.model_selection import train_test_split

ft = pd.read_parquet(FT_POOL)
strata = (ft['dataset_role'].astype(str) + '|'
          + ft['head_mask_lattice'].astype(str)
          + ft['head_mask_spacegroup'].astype(str)
          + ft['head_mask_crystal_system'].astype(str)
          + ft['head_mask_elements'].astype(str)).to_numpy()
tr_i, va_i = train_test_split(np.arange(len(ft)), test_size=0.05,
                              random_state=SEED, stratify=strata)
ft['ft_split'] = 'train'
ft.loc[ft.index[va_i], 'ft_split'] = 'val'
ft.to_parquet(OUT / 'splits_ft.parquet')
print(ft.groupby(['dataset_role', 'ft_split']).size().to_string())
print()
print('валидация FT: всего', int((ft['ft_split'] == 'val').sum()), 'строк — метрики будут шумными, это плата за 95/5')

dataset_role  ft_split
opxrd         train       2010
              val          106
rruff         train       1291
              val           68

валидация FT: всего 174 строк — метрики будут шумными, это плата за 95/5
